# 用 Phi-3 生成文字：載入現成的指令模型並提問

---

## 這份程式要做什麼

載入微軟的小型語言模型 **Phi-3-mini**，丟給它一個問題，讓它回答：

```
問題：  "Create a funny joke about chicken?"
          │
          ▼
   Phi-3-mini-4k-instruct（已經訓練好的聊天模型）
          │
          ▼
回答：  一則關於雞的笑話
```

這份程式**沒有訓練任何東西**，只是**使用**一個已經訓練好的模型，這個過程叫**推論（inference）**。

## 和前面學過的內容的關係

| 階段 | 做什麼 | 對應之前的程式 |
|---|---|---|
| 預訓練 | 在海量文字上學會語言 | 別人做好了 |
| 微調（SFT、DPO） | 學會對話、學會偏好 | QLoRA、DPO 那幾份 |
| **推論（這份）** | **拿訓練好的模型來用** | 之前每份最後面的 `pipeline` 生成 |

之前每份程式最後都有一段 `pipeline("text-generation", ...)`，這份就是把那一段**單獨拿出來**，用在一個別人已經訓練好的模型上。

## 先搞懂三個關鍵詞

### 1. 因果語言模型（Causal LM）與逐字生成

這類模型每次只做一件事：**根據前面所有的字，預測下一個字**。生成一整段文字，就是不斷重複這件事：

```
輸入： Why did the chicken
預測： cross        → Why did the chicken cross
預測： the          → Why did the chicken cross the
預測： road         → Why did the chicken cross the road
...直到產生結束符號，或達到字數上限
```

### 2. 指令模型（Instruct）

| | 基礎模型（base） | 指令模型（instruct） |
|---|---|---|
| 訓練目標 | 預測下一個詞 | 在此之上，再訓練成「能依指示回答」 |
| 給它一個問題 | 可能接著寫出更多問題 | 給出回答 |

名稱裡的 **instruct** 代表它已經做過類似 SFT 的指令微調，所以可以直接拿來對話。

### 3. 名稱 `Phi-3-mini-4k-instruct` 的意思

| 部分 | 意思 |
|---|---|
| `Phi-3` | 微軟的 Phi 系列第三代 |
| `mini` | 小型版本，我印象中約 38 億參數 |
| `4k` | 上下文長度約 4096 個 token（一次能看的文字上限） |
| `instruct` | 指令微調版 |

## 整體流程

```
載入模型 → 載入分詞器 → 建立生成管線（設定生成規則）→ 準備訊息 → 生成並印出
```

## 步驟 0：載入套件

| 匯入 | 用途 |
|---|---|
| `AutoModelForCausalLM` | 自動載入因果語言模型（逐字預測下一個詞的模型） |
| `AutoTokenizer` | 自動載入配對的分詞器 |
| `pipeline` | 把「分詞 → 生成 → 還原成文字」包成一個簡單的工具 |

名稱裡的 **Auto** 的意思是：你只給模型名稱，它會自動判斷該用哪一種模型架構。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline

## 步驟 1：載入模型

```python
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
)
```

| 參數 | 說明 |
|---|---|
| `"microsoft/Phi-3-mini-4k-instruct"` | 模型在 Hugging Face 上的名稱。第一次執行會自動下載，之後從本機快取讀取 |
| `device_map="cuda"` | 把**整個模型放到 GPU** 上 |
| `torch_dtype="auto"` | 依模型設定檔裡記錄的精度載入（通常是 16 位元），不要預設成 32 位元 |

### `device_map` 的幾種選擇

| 值 | 效果 |
|---|---|
| `"cuda"` | 整個模型放 GPU，放不下就報錯 |
| `"auto"` | 自動分配，GPU 放不下的部分放到 CPU（較慢，但不會直接失敗） |
| `"cpu"` | 全部放 CPU（很慢） |

### `torch_dtype="auto"` 為什麼重要

| 精度 | 每個參數佔用 | 3.8B 參數的模型 |
|---|---|---|
| 32 位元 | 4 bytes | 約 15 GB |
| **16 位元（auto 通常選這個）** | **2 bytes** | **約 7.6 GB** |

用 `auto` 可以讓記憶體用量直接減半。

> 版本提醒：在較新版本的 transformers 裡，`torch_dtype` 可能改名為 `dtype`，舊名稱可能會出現警告。我沒有針對你的版本確認。

### ⚠️ 顯存提醒

模型權重本身就約 7.6 GB（我用參數量乘上每個參數的位元組數估算的）。**8 GB 的顯卡可能放不下**，再加上生成時需要的暫存空間，很容易 `CUDA out of memory`。解法：

1. 改用 `device_map="auto"`，讓放不下的部分自動放到 CPU
2. 改用 4 位元量化載入（你在 QLoRA 學過的 `BitsAndBytesConfig`），記憶體只需要約 1/4（做法見最後的實驗）

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",                
    )

## 步驟 2：載入分詞器

```python
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
```

分詞器負責**文字和數字之間的轉換**：

```
"Create a funny joke"  ──分詞──▶  [token id, token id, ...]  ──模型──▶  [新的 token id, ...]  ──還原──▶  文字
```

**模型和分詞器必須配對**，所以兩邊都用同一個名稱。

### 分詞器還有一個重要任務：對話模板

指令模型是在**特定的對話格式**下訓練的。這個格式存在分詞器的設定裡，叫「對話模板」（chat template）。Phi-3 的格式大致像這樣（依我的印象，細節以實際模板為準）：

```
<|user|>
Create a funny joke about chicken?<|end|>
<|assistant|>
```

模型看到 `<|assistant|>`，就知道「現在輪到我回答」。後面會看到，`pipeline` 會**自動幫你套用這個模板**。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
)

## 步驟 3：建立生成管線（設定生成規則）

```python
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False,
)
```

### `pipeline` 做了什麼

把一整串步驟包成一個工具：

```
你的訊息 → ① 套用對話模板 → ② 分詞 → ③ 模型逐字生成 → ④ 還原成文字 → ⑤ 去掉提示詞 → 回答
```

### 參數說明

| 參數 | 值 | 說明 |
|---|---|---|
| `"text-generation"` | — | 任務類型：文字生成 |
| `model`、`tokenizer` | 前面載入的 | 傳入已載入的模型與分詞器 |
| `return_full_text=False` | False | **只回傳模型生成的部分**，不包含你的提示詞。設成 True 的話，回傳會連提示詞一起 |
| `max_new_tokens=500` | 500 | **最多生成 500 個新 token**。遇到結束符號會提前停止，不一定會用滿 |
| `do_sample=False` | False | **不使用隨機取樣**，每一步都選機率最高的字（貪婪解碼） |

### `max_new_tokens` 與 `max_length` 的差別

| 參數 | 計算範圍 |
|---|---|
| `max_new_tokens` | 只算**新生成**的部分 |
| `max_length` | 算**提示詞 + 生成**的總長度 |

用 `max_new_tokens` 比較直觀，不用先算提示詞有多長。

### `do_sample` 是什麼

模型每一步會對詞彙表裡每個字算出一個機率，接著要決定選哪一個：

| `do_sample` | 做法 | 結果 |
|---|---|---|
| **`False`（這份程式）** | 永遠選機率最高的 | **確定性**：同樣的問題，每次回答一模一樣 |
| `True` | 依機率隨機抽 | 每次回答不同，更有創意，也更容易跑偏 |

貪婪解碼的缺點是有時會**重複**或顯得呆板；優點是結果可重現，適合要穩定輸出的情況。

### 想要有變化的話（要自己加參數，原程式沒有）

當 `do_sample=True` 時，以下參數才有作用：

| 參數 | 作用 |
|---|---|
| `temperature` | 越高越隨機，越低越保守（接近貪婪） |
| `top_k` | 只從機率最高的 k 個字中抽 |
| `top_p` | 只從累積機率達 p 的最小字集合中抽 |

> 注意：`do_sample=False` 時，這些參數會被忽略（可能會有警告）。

In [ ]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,    #只回傳結果 不包含提示詞
    max_new_tokens=500,
    do_sample=False,           #模型是否使用採樣策略選擇下一個字元
)

## 步驟 4：準備訊息

```python
message = [
    {"role": "user", "content": "Create a funny joke about chicken?"}
]
```

### 訊息的格式

對話用「**訊息清單**」表示：一個 list，每則訊息是一個 dict，有兩個欄位：

| 欄位 | 說明 |
|---|---|
| `role` | **誰說的**：`"user"`（使用者）、`"assistant"`（助理）、`"system"`（系統提示） |
| `content` | **說了什麼** |

這和你之前 SFT 資料集（UltraChat）的 `messages` 格式是同一種。

### 為什麼這樣傳，而不是直接傳字串

因為 `pipeline` 看到的是「訊息清單」，它會**自動用分詞器的對話模板**，把訊息轉成模型認得的格式（加上 `<|user|>`、`<|assistant|>` 等標記）。你不需要自己拼字串。

之前 DPO 那份程式的最後，是自己手動拼：

```python
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""
```

這份則是交給 pipeline 處理，**不容易因為空白、換行拼錯**。

### 多輪對話與系統提示

想要多輪對話，就把訊息接長：

```python
message = [
    {"role": "system",    "content": "You are a witty comedian."},
    {"role": "user",      "content": "Tell me a joke about chicken."},
    {"role": "assistant", "content": "Why did the chicken join a band? ..."},
    {"role": "user",      "content": "Another one, please."},
]
```

In [ ]:
message = [
    {"role": "user", "content": "Create a funny joke about chicken?"}
]

## 步驟 5：生成並印出結果

```python
output = generator(message)
print(output[0]['generated_text'])
```

### 輸出的結構

`generator(...)` 回傳的是**一個 list**，裡面每個元素是一個 dict：

```python
[{'generated_text': '這裡是模型生成的笑話……'}]
```

| 寫法 | 意思 |
|---|---|
| `output` | 整個 list |
| `output[0]` | 第一個結果（這裡只有一個） |
| `output[0]['generated_text']` | 第一個結果裡的生成文字 |

### 為什麼 `generated_text` 是純文字

因為 `return_full_text=False`，所以只有模型新生成的那部分。

如果設成 `return_full_text=True`，而且輸入是訊息清單，我印象中 `generated_text` 會變成**整段對話的訊息清單**（包含你的提問與助理的回答），而不是字串。這點我沒有針對你的版本確認。

### 因為是貪婪解碼，結果會一樣

`do_sample=False` 的情況下，同樣的問題、同樣的模型，**每次執行都會得到相同的笑話**。想看到不同的結果，要打開取樣（見步驟 3 的說明）。

In [ ]:
output = generator(message)
print(output[0]['generated_text'])

---

# 重點整理

## 一、整份程式一句話

> 載入已經訓練好的指令模型 Phi-3-mini 與其分詞器，用 `pipeline` 設定「貪婪解碼、最多生成 500 個新 token、只回傳生成部分」，丟入一則使用者訊息，印出模型的回答。

## 二、必須搞懂的 10 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | 推論（inference） | 使用訓練好的模型，不再更新權重 |
| 2 | 因果語言模型 | 根據前面的字，逐字預測下一個字 |
| 3 | 指令模型（instruct） | 在基礎模型上做過指令微調，能依指示回答 |
| 4 | 分詞器 | 文字與數字之間的轉換，必須和模型配對 |
| 5 | 對話模板 | 存在分詞器裡的格式，標明誰在說話；pipeline 會自動套用 |
| 6 | 訊息清單 | `[{"role": ..., "content": ...}]`，表示一段對話 |
| 7 | `max_new_tokens` | 最多生成多少個新 token，只算新生成的部分 |
| 8 | `do_sample=False` | 貪婪解碼，每步選機率最高的字，結果固定 |
| 9 | `return_full_text` | 是否把提示詞也一起回傳 |
| 10 | `device_map` / `torch_dtype` | 決定模型放哪個裝置、用什麼精度，直接影響記憶體 |

## 三、程式流程速記

```
1. from_pretrained（模型）  → 放到 GPU，用模型本身記錄的精度
2. from_pretrained（分詞器）→ 配對的分詞器，附帶對話模板
3. pipeline("text-generation") → 設定 return_full_text / max_new_tokens / do_sample
4. message                  → 訊息清單，role + content
5. generator(message)       → 套模板 → 分詞 → 生成 → 還原 → 去掉提示詞
6. output[0]['generated_text'] → 取出回答
```

## 四、最容易搞混的地方

1. **這份程式有訓練嗎？**
   沒有。它只使用已經訓練好的模型，叫推論。

2. **為什麼要傳訊息清單，不是字串？**
   pipeline 會用分詞器的對話模板自動組成正確格式，不用自己拼 `<|user|>` 這些標記。

3. **`max_new_tokens=500` 是一定會生成 500 個嗎？**
   不是。是「最多」，遇到結束符號就提前停止。

4. **為什麼每次結果都一樣？**
   `do_sample=False` 是貪婪解碼，沒有隨機性。

5. **`do_sample=False` 時加 `temperature` 有用嗎？**
   沒有，這些取樣參數只在 `do_sample=True` 時才生效。

6. **`return_full_text=False` 回傳什麼？**
   只有模型新生成的部分，不含你的提問。

7. **`torch_dtype="auto"` 為什麼重要？**
   用模型原本記錄的 16 位元載入，比預設的 32 位元省一半記憶體。

8. **模型一定要放 GPU 嗎？**
   不一定，但 CPU 會慢很多。`device_map="auto"` 可以讓放不下的部分自動放到 CPU。

## 五、常見錯誤排查表

| 症狀 | 可能原因 | 處理 |
|---|---|---|
| `CUDA out of memory` | 模型約 7.6 GB，顯存不足 | 改 `device_map="auto"`，或用 4 位元量化載入 |
| 載入時卡很久 | 第一次執行，正在下載模型（數 GB） | 等待下載完成，之後會從快取讀取 |
| `torch_dtype` 的警告 | 新版 transformers 改名為 `dtype` | 依警告改名即可 |
| 每次回答都一樣 | `do_sample=False` | 改成 `do_sample=True` 並設定 `temperature` |
| 回答重複、呆板 | 貪婪解碼的特性 | 開取樣，或加 `repetition_penalty` |
| 回答被截斷 | 達到 `max_new_tokens` 上限 | 調大 `max_new_tokens` |
| 沒有 GPU 卻用 `device_map="cuda"` | 找不到 CUDA 裝置 | 改 `"cpu"` 或 `"auto"` |

## 六、觀察到的幾個地方（程式碼未修改，僅供參考）

1. **`device_map="cuda"` 對 8 GB 顯卡可能不夠**
   模型權重約 7.6 GB，我是用參數量估算的，沒有實測。

2. **`torch_dtype` 在新版可能改名為 `dtype`**
   舊名稱可能還能用，但會有警告。

3. **貪婪解碼讓輸出固定**
   適合要可重現的場合，不適合要多樣化創意的情境（例如寫笑話）。

4. **模型與分詞器分別載入**
   兩邊的名稱要一致，用錯配對會得到亂碼或品質很差的結果。

5. **只問了一個問題**
   無法判斷模型的整體品質，建議多測幾個。

6. **沒有系統提示**
   模型的行為完全依賴預設，加上 `system` 訊息可以控制語氣與角色。

## 七、可以自己做的實驗（需要時再自己加，不影響原程式）

```python
# 1. 看對話模板長什麼樣（pipeline 在背後套用的就是這個）
print(tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True))

# 2. 打開取樣，每次得到不同的笑話
generator = pipeline("text-generation", model=model, tokenizer=tokenizer,
                     return_full_text=False, max_new_tokens=200,
                     do_sample=True, temperature=0.8, top_p=0.9)
print(generator(message)[0]["generated_text"])

# 3. 加上系統提示，改變語氣
message = [
    {"role": "system", "content": "You are a sarcastic comedian."},
    {"role": "user",   "content": "Create a funny joke about chicken?"},
]

# 4. 數一數提示詞和回答各有幾個 token
ids = tokenizer.apply_chat_template(message, add_generation_prompt=True)
print(len(ids))

# 5. 用 4 位元量化載入，大幅降低記憶體（需要 bitsandbytes 與 NVIDIA GPU）
from transformers import BitsAndBytesConfig
import torch
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
model_4bit = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct", quantization_config=bnb, device_map="auto")

# 6. 多輪對話：把上一輪的回答放進訊息清單，再問下一題
```

## 八、與前面學過的內容對照

| | 微調（QLoRA、DPO） | **推論（這份）** |
|---|---|---|
| 目的 | 改變模型的權重 | 使用模型生成文字 |
| 需要訓練資料 | 要 | **不要** |
| 需要的顯存 | 訓練時較高 | 只需放得下模型與生成暫存 |
| 對話格式 | 自己用 `apply_chat_template` 或手動拼接 | **pipeline 自動套用模板** |
| 關鍵參數 | 學習率、batch size、LoRA 的 `r` | **`max_new_tokens`、`do_sample`、`temperature`** |

## 九、一句話記住

> **推論 = 模型 + 分詞器 + 生成規則。** 你只需要告訴 pipeline 要用哪個模型、最多寫多長、要不要隨機，剩下的（套對話模板、分詞、逐字生成、還原文字）它都會替你處理。